In [1]:
import pandas as pd
import igraph
import pickle as pk
import os

In [2]:
os.listdir("data/")

['LitLangMed_enriched.csv', 'Humanities_enriched.csv', 'Science_enriched.csv']

In [3]:
dataset = 'LitLangMed'

In [4]:
df = pd.read_csv( "data/" + dataset + "_enriched.csv", dtype={43:str, 44:str, 45:str, 46:str} )
print( df.shape )
print( df.columns )

(7017, 60)
Index(['School', 'Folder', 'Document Name', 'Thesis Title', 'Thesis Author',
       'Citation ID', 'Citation Type', 'Citation Title', 'Citation Authors',
       'Citation Editors', 'Citation Year', 'Journal', 'Publisher',
       'Citation Source', 'Place', 'Series', 'Volume', 'Issue', 'Pages', 'DOI',
       'Reference URL', 'Note', '_is_duplicate_within_thesis', '_is_noise',
       '_noise_reason', '_citation_type_original', 'eligible_for_matching',
       '_ineligible_reason', 'Thesis Year', 'Citation Type (refined)',
       'Citation Author Count', 'Citation Is Single Author', 'ref_id',
       'openalex_status', 'openalex_affiliations', 'openalex_countries',
       'openalex_author_breakdown', 'openalex_author_country_pairs',
       'openalex_matched_author_count', 'openalex_main_author_matched',
       'openalex_distinct_author_countries',
       'openalex_is_international_collab', 'pyorcid_status',
       'pyorcid_affiliations', 'pyorcid_countries', 'pyorcid_author_break

In [5]:
df

,School,Folder,Document Name,Thesis Title,Thesis Author,Citation ID,Citation Type,Citation Title,Citation Authors,Citation Editors,...,pyorcid_is_international_collab,crossref_status,crossref_affiliations,crossref_countries,crossref_author_breakdown,crossref_author_country_pairs,crossref_matched_author_count,crossref_main_author_matched,crossref_distinct_author_countries,crossref_is_international_collab
0,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,1,Web/Grey Literature,Rethinking Public Service Broadcasting's Place...,S Abbott,NaN,...,False,hit_no_match,NaN,NaN,NaN,NaN,0,False,0,False
1,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,2,Book Chapter,Media and Cultural Industries,T Adorno; M Horkheimer,M Kellner; & Durham,...,False,hit_no_match,NaN,NaN,NaN,NaN,0,False,0,False
2,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,3,Web/Grey Literature,Load shedding gives rise to growth in SA Strea...,L Adams,NaN,...,False,hit_no_match,NaN,NaN,NaN,NaN,0,False,0,False
3,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,4,Journal Article,Disability and sexuality: Claiming sexual and ...,R Addlakha; J Price; S Heidari,NaN,...,False,matched_no_affiliation,NaN,NaN,NaN,NaN,3,True,0,False
4,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,5,Journal Article,A Theory of the Public Sphere,A Adut,NaN,...,False,matched_with_affiliation,"University of Texas at Austin, Austin, TX, USA",USA,"Ari Adut: University of Texas at Austin, Austi...",Ari Adut|USA,1,True,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7012,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,681,Journal Article,Political effects of the internet and social m...,E Zhuravskaya; M Petrova; R Enikolopov,NaN,...,False,matched_no_affiliation,NaN,NaN,NaN,NaN,3,True,0,False
7013,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,682,Journal Article,International dimensions of electoral processe...,C Ziegler,NaN,...,False,matched_no_affiliation,NaN,NaN,NaN,NaN,1,True,0,False
7014,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,683,Journal Article,"Tweet, but verify: Epistemic study of informat...",A Zubiaga; H Ji,NaN,...,False,matched_no_affiliation,NaN,NaN,NaN,NaN,2,True,0,False
7015,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,684,Other,1708153@students.wits.ac,Samuel Yours Sincerely; Olaniran,NaN,...,False,hit_no_match,NaN,NaN,NaN,NaN,0,False,0,False


In [6]:
def create_graph_data(df):
    
    ## CREATE NODES

    # Get unique phd documents based on doc name
    df_phds = df.drop_duplicates(subset=['Document Name'])[['Document Name', 'School']]
    df_phds.columns = ['title', 'school']
    df_phds['type'] = 'Wits PhD'

    # Get unique refs based on title, authors and ref type
    df_refs = df.drop_duplicates(subset=['Citation Authors'])[['Citation Authors', 'Citation Type']]
    df_refs.columns = ['authors', 'type']

    # Concatenate into single node df and generate unique id for each node
    df_nodes = pd.concat( [
        df_phds, 
        df_refs
    ], ignore_index = True).reset_index(names='id')

    
    ## CREATE EDGES

    # Create mappings from phd docs and refs to node ids
    phd_map = df_nodes[df_nodes['type'] == 'Wits PhD'].set_index('title')['id']    
    ref_map = df_nodes[df_nodes['type'] != 'Wits PhD'].set_index(['authors'])['id']

    # Create edges by mapping phd doc names into source column, and ref title, author and data into target column
    df_edges = pd.DataFrame({
        'source': df['Document Name'].map(phd_map),
        'target': df['Citation Authors'].map(ref_map)
    })

    return df_nodes, df_edges

def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [7]:
nodes, edges = create_graph_data(df)

In [8]:
g = igraph.Graph.DataFrame(edges, directed=True, vertices=nodes, use_vids=False)
describe(g)

5298
7017
['name', 'title', 'school', 'type', 'authors']
[]


In [9]:
## Write graph to pickle
g.write_pickle( "graphs/" + dataset + "_graph.pk")